# 1. Phase 3 Random Forest Baseline

In [1]:
# Standard Data Science Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure plots show up inline
%matplotlib inline

In [2]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score


In [3]:
# 1.Load your dataset
df = pd.read_csv('final_model_ready_data.csv')

In [4]:
for col in sorted(df.columns):
    print(col)

1stFlrSF
2ndFlrSF
3SsnPorch
BedroomAbvGr
BldgType
BldgType_2fmCon
BldgType_Duplex
BldgType_Twnhs
BldgType_TwnhsE
BsmtCond
BsmtCond_Gd
BsmtCond_Po
BsmtCond_TA
BsmtExposure
BsmtExposure_Gd
BsmtExposure_Mn
BsmtExposure_No
BsmtFinSF1
BsmtFinSF2
BsmtFinType1
BsmtFinType1_BLQ
BsmtFinType1_GLQ
BsmtFinType1_LwQ
BsmtFinType1_Rec
BsmtFinType1_Unf
BsmtFinType2
BsmtFinType2_BLQ
BsmtFinType2_GLQ
BsmtFinType2_LwQ
BsmtFinType2_Rec
BsmtFinType2_Unf
BsmtFullBath
BsmtHalfBath
BsmtQual
BsmtQual_Fa
BsmtQual_Gd
BsmtQual_TA
BsmtUnfSF
CentralAir
CentralAir_Y
Condition1
Condition1_Feedr
Condition1_Norm
Condition1_PosA
Condition1_PosN
Condition1_RRAe
Condition1_RRAn
Condition1_RRNe
Condition1_RRNn
Condition2
Condition2_Feedr
Condition2_Norm
Condition2_PosA
Condition2_PosN
Condition2_RRAe
Condition2_RRAn
Condition2_RRNn
Electrical
Electrical_FuseF
Electrical_FuseP
Electrical_Mix
Electrical_SBrkr
EnclosedPorch
ExterCond
ExterCond_Fa
ExterCond_Gd
ExterCond_Po
ExterCond_TA
ExterQual
ExterQual_Fa
ExterQual_Gd
Exter

In [5]:
# 2. Select only numeric columns (This automatically handles any remaining text)
df_numeric = df.select_dtypes(exclude=['object'])

In [6]:
# 3. Define Features (X) and Target (y)
X = df_numeric.drop('SalePrice', axis=1)
y = df_numeric['SalePrice']

In [7]:
# 4. Split data (Using random_state=42 for consistency with Iffah)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
# 5. Initialize and Train Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [9]:
# 6. Evaluate
predictions = rf.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print(f"Random Forest Baseline: RMSE={rmse:.2f}, R2={r2:.2f}")

Random Forest Baseline: RMSE=28897.07, R2=0.89


#### That is a solid start for our baseline model! An $R^2$ of 0.89 means that your Random Forest model is already explaining 89% of the variance in housing prices, which is a very strong performance for a first run.  

### **What these numbers tell us:**
#### $R^2$ (0.89): Our model is capturing the vast majority of the "price signals" in our data. Anything above 0.85 is generally considered very good for residential real estate valuation.  

#### RMSE (28,897.07): On average, our model's predictions are off by about $28,897. In the context of housing, this is the "typical error" in our dollar valuation. 

# 2. Phase 4: Model Tuning with RandomizedSearchCV

In [10]:
from sklearn.model_selection import RandomizedSearchCV

In [11]:
# 1. Define the grid of parameters to test
param_dist = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10]
}

In [12]:
# 2. Initialize the search
# n_iter=10 tries 10 random combinations; cv=5 uses 5-fold cross-validation
random_search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_distributions=param_dist,
    n_iter=10,
    cv=5, 
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=-1
)

In [13]:
# 3. Fit to the training data
print("Starting hyperparameter tuning...")
random_search.fit(X_train, y_train)

Starting hyperparameter tuning...


,estimator,RandomForestR...ndom_state=42)
,param_distributions,"{'max_depth': [None, 10, ...], 'min_samples_split': [2, 5, ...], 'n_estimators': [50, 100, ...]}"
,n_iter,10
,scoring,'neg_root_mean_squared_error'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [14]:
# 4. Extract the best model
best_rf = random_search.best_estimator_
print(f"Best Parameters: {random_search.best_params_}")

Best Parameters: {'n_estimators': 100, 'min_samples_split': 5, 'max_depth': 30}


In [15]:
# 5. Predict and calculate metrics
final_predictions = best_rf.predict(X_test)

final_rmse = np.sqrt(mean_squared_error(y_test, final_predictions))
final_r2 = r2_score(y_test, final_predictions)

In [16]:
# 6. Print final optimized results
print(f"Optimized RMSE: {final_rmse:.2f}")
print(f"Optimized R2: {final_r2:.4f}")

Optimized RMSE: 29169.32
Optimized R2: 0.8891


# 3. Model Performance Comparison

In [17]:
print("--- Model Performance Comparison ---")
print(f"Baseline RMSE:  {rmse:.2f}")
print(f"Optimized RMSE: {final_rmse:.2f}")
print(f"Difference:     {final_rmse - rmse:.2f}")
print("-" * 35)
print(f"Baseline R2:    {r2:.4f}")
print(f"Optimized R2:   {final_r2:.4f}")
print(f"Difference:     {final_r2 - r2:.4f}")

--- Model Performance Comparison ---
Baseline RMSE:  28897.07
Optimized RMSE: 29169.32
Difference:     272.25
-----------------------------------
Baseline R2:    0.8911
Optimized R2:   0.8891
Difference:     -0.0021


#### **RMSE Difference:** If the result is positive (like +272.25), it means your optimized model had a slightly higher error than the baseline.
#### **$R^2$ Difference:** If the result is negative, it means our optimized model explained slightly less variance than your baseline.

#### This approach gives us a very clear, numerical way to report that the baseline was indeed the better choice for your project. 

# Comparison List (Actual vs. Predicted)

In [22]:
# Create a DataFrame to view the numbers side-by-side
comparison_list = pd.DataFrame({
    'Actual Price': y_test,
    'Predicted Price': predictions,
    'Difference (Error)': y_test - predictions
})

# Show the first 10 rows as a list
print(comparison_list.head(10).to_string())

      Actual Price  Predicted Price  Difference (Error)
892         154500        140576.75            13923.25
1105        325000        324646.13              353.87
413         115000        118666.50            -3666.50
522         159000        155072.20             3927.80
1036        315500        322145.30            -6645.30
614          75500         85529.50           -10029.50
218         311500        211030.06           100469.94
1160        146000        149471.93            -3471.93
649          84500         84456.99               43.01
887         135500        131513.74             3986.26


### **What these numbers represent:** 

#### **Actual Price:** This is the real sale price of the house in your dataset.  
#### **Predicted Price:** This is what your Random Forest model estimated the price to be.
#### **Difference(Error):** This number shows you exactly how much your model missed by.  If the number is positive, your model underestimated the price.  If the number is negative, your model overestimated the price. 
#### **Accuracy:** House 1105 or 649, the model works perfectly.
#### **Limitations:** While the model is highly accurate for most properties, it occasionally struggles with unique outliers, such as house 218, which may have characteristics that fall outside of standard patterns.